In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.metrics.pairwise import haversine_distances
from math import radians
import requests
import json

In [2]:
#Read the json file of MRt Exit by LTA
# https://github.com/xkjyeah/MRT-and-LRT-Stations/blob/master/mrt_lrt.csv
mrtDF = pd.read_csv('Data\mrt_lrt.csv') #Read MRT Coords
addressDF = pd.read_csv('SNRE_Dataset_2024_Outliers_feb2024.csv') # Read Dataset
display(addressDF.head(5))

,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Type of Sale,Region,Remaining Tenure (Years),Total Population,SORA (%),...,Rental Index,Rental Index % change,Location,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,lagged psf
0,2020-03-12,VERTEX,UBI AVENUE 3,1561.0,Non-First Floor,Resale,Central Region,47,5685807.0,0.27,...,89.9,-10.1,Central Region-Geylang-14-40,Geylang,14,40,2020 1Q,3,2020,474.08
1,2016-03-23,WEST CONNECT BUILDING,BUROH STREET,1733.0,First Floor,New Sale,West Region,27,5607283.0,0.13,...,88.8,-11.2,West Region-Boon Lay-22-62,Boon Lay,22,62,2016 1Q,3,2016,400.43
2,2015-06-05,NORDCOM ONE,GAMBAS CRESCENT,1690.0,Non-First Floor,New Sale,North Region,29,5535002.0,1.01,...,98.4,-1.6,North Region-Sembawang-27-75,Sembawang,27,75,2015 2Q,6,2015,332.64
3,2014-09-30,CT HUB 2,LAVENDER STREET,1109.0,Non-First Floor,New Sale,Central Region,61,5469724.0,0.08,...,99.7,-0.3,Central Region-Kallang-12-33,Kallang,12,33,2014 3Q,9,2014,613.27
4,2017-10-25,UBI TECHPARK,UBI CRESCENT,2605.0,Non-First Floor,Resale,Central Region,40,5612253.0,1.10,...,91.8,-8.2,Central Region-Geylang-14-40,Geylang,14,40,2017 4Q,10,2017,566.37


In [3]:
# i = 'JURONG PORT ROAD'

# url = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y".format(i)
# response = requests.request("GET", url)
# print(response.text)

In [4]:
# https://www.onemap.gov.sg/
def geocodeProperty(list):
    nameList,latList, longList = [], [], []
    for i in list:
        proj_name= i[0]
        nameList.append(proj_name)
        url_proj = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y".format(proj_name)
        proj_response = requests.request("GET", url_proj)
        projDict = json.loads(proj_response.text)
        
        street_name = i[1]
        url_street = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y".format(street_name)
        street_response = requests.request("GET", url_street)
        streetDict = json.loads(street_response.text)
               
        if projDict['found'] == 0:
            latList.append(streetDict['results'][0]['LATITUDE'])
            longList.append(streetDict['results'][0]['LONGITUDE'])
        else:
            latList.append(projDict['results'][0]['LATITUDE'])
            longList.append(projDict['results'][0]['LONGITUDE'])
    geoCodeDF = pd.DataFrame({'Project Name':nameList,
                              'latitude':pd.Series(latList,name = 'latitude'),
                              'longitude':pd.Series(longList,name = 'longitude')})    
    return geoCodeDF


In [5]:
nameDF = addressDF[['Project Name','Street Name']].drop_duplicates('Project Name').reset_index(drop=True)
nameList = nameDF.values.tolist() # a Total of 138 unique building names, caution of "Woodlands east industrial estate"
display(len(nameList))
propertyGeoCodeDF = geocodeProperty(nameList)
propertyGeoCodeDF = propertyGeoCodeDF.astype({'latitude':'float64',
                                              'longitude':'float64'})

display(propertyGeoCodeDF)
display(propertyGeoCodeDF.info())


138

,Project Name,latitude,longitude
0,VERTEX,1.332960,103.894116
1,WEST CONNECT BUILDING,1.312530,103.702980
2,NORDCOM ONE,1.444080,103.813889
3,CT HUB 2,1.311588,103.863375
4,UBI TECHPARK,1.326337,103.896262
...,...,...,...
133,NORTHLAND INDUSTRIAL BUILDING 1,1.452881,103.794669
134,LOKYANG LIGHT INDUSTRIES PARK,1.335107,103.696439
135,CRESCENDAS PRINT MEDIA HUB,1.337761,103.890570
136,SKYTECH,1.342143,103.754379


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Project Name  138 non-null    object 
 1   latitude      138 non-null    float64
 2   longitude     138 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.4+ KB


None

In [6]:
display(propertyGeoCodeDF)

,Project Name,latitude,longitude
0,VERTEX,1.332960,103.894116
1,WEST CONNECT BUILDING,1.312530,103.702980
2,NORDCOM ONE,1.444080,103.813889
3,CT HUB 2,1.311588,103.863375
4,UBI TECHPARK,1.326337,103.896262
...,...,...,...
133,NORTHLAND INDUSTRIAL BUILDING 1,1.452881,103.794669
134,LOKYANG LIGHT INDUSTRIES PARK,1.335107,103.696439
135,CRESCENDAS PRINT MEDIA HUB,1.337761,103.890570
136,SKYTECH,1.342143,103.754379


In [7]:
mrtDF = mrtDF[['Name','Latitude','Longitude']]
mrtDF.rename(columns={'Name':'mrt_station',
                      'Latitude':'latitude',
                      'Longitude':'longitude'}, inplace=True)
display(mrtDF)
display(mrtDF.info())

,mrt_station,latitude,longitude
0,ADMIRALTY MRT STATION,1.440585,103.800988
1,ALJUNIED MRT STATION,1.316433,103.882906
2,ANG MO KIO MRT STATION,1.369933,103.849558
3,BARTLEY MRT STATION,1.342501,103.880178
4,BAYFRONT MRT STATION,1.281874,103.859080
...,...,...,...
188,TECK WHYE LRT STATION,1.376685,103.753712
189,TEN MILE JUNCTION LRT STATION,1.380321,103.760140
190,THANGGAM LRT STATION,1.397318,103.875635
191,TONGKANG LRT STATION,1.389348,103.885844


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   mrt_station  193 non-null    object 
 1   latitude     193 non-null    float64
 2   longitude    193 non-null    float64
dtypes: float64(2), object(1)
memory usage: 4.6+ KB


None

In [8]:
propertyGeoCodeDF.loc[0,['latitude','longitude']]

latitude        1.33296
longitude    103.894116
Name: 0, dtype: object

In [9]:
# # Computing the distance using haversine_distances
# # The Haversine (or great circle) distance is the angular distance between two points on the surface of a sphere. 
# # The first coordinate of each point is assumed to be the latitude, the second is the longitude, given in radians. 
# # The dimension of the data must be 2
# # 138 Buildings, 193 MRT & LRT station

# Use a loop structure to compute the nearest mrt distance
distList, minDistList = [], []
minDistDF = pd.DataFrame()
for i in range(propertyGeoCodeDF.shape[0]): # Outer loop to loop through all the buildings, from propertyGeoCodeDF
    propCoord = propertyGeoCodeDF.loc[i,['latitude','longitude']]
    propCoord_in_rad = [radians(_) for _ in propCoord]
    for i in range(mrtDF.shape[0]): # Inner loop to loop through all the mrt exits
        mrtCoord = mrtDF.loc[i,['latitude','longitude']]
        mrtDist_in_rad = [radians(_) for _ in mrtCoord] #Convert to radian according to docs
        result = haversine_distances([mrtDist_in_rad, propCoord_in_rad])
        resultInKM = (result * 6371)
        distList.append(resultInKM[0,1].round(5))
    minDistList.append(min(distList))
    distList = []
display(len(minDistList))


138

In [10]:
minDistToMRT = pd.Series(minDistList, name = 'minDistToMRT')
propertyGeoCodeDF = pd.concat([propertyGeoCodeDF, minDistToMRT], axis = 1)
propertyGeoCodeDF['minDistToMRT'] = (propertyGeoCodeDF['minDistToMRT'] * 1000)


In [11]:
display(propertyGeoCodeDF)

,Project Name,latitude,longitude,minDistToMRT
0,VERTEX,1.332960,103.894116,657.96
1,WEST CONNECT BUILDING,1.312530,103.702980,2856.34
2,NORDCOM ONE,1.444080,103.813889,879.74
3,CT HUB 2,1.311588,103.863375,235.93
4,UBI TECHPARK,1.326337,103.896262,521.75
...,...,...,...,...
133,NORTHLAND INDUSTRIAL BUILDING 1,1.452881,103.794669,1089.52
134,LOKYANG LIGHT INDUSTRIES PARK,1.335107,103.696439,292.70
135,CRESCENDAS PRINT MEDIA HUB,1.337761,103.890570,379.27
136,SKYTECH,1.342143,103.754379,934.51


In [12]:
addressDF1 = addressDF.merge(propertyGeoCodeDF, how = 'inner', left_on='Project Name', right_on='Project Name', sort=False)

In [13]:
display(addressDF1.head(5))

,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Type of Sale,Region,Remaining Tenure (Years),Total Population,SORA (%),...,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,lagged psf,latitude,longitude,minDistToMRT
0,2020-03-12,VERTEX,UBI AVENUE 3,1561.0,Non-First Floor,Resale,Central Region,47,5685807.0,0.27,...,Geylang,14,40,2020 1Q,3,2020,474.08,1.33296,103.894116,657.96
1,2021-12-28,VERTEX,UBI AVENUE 3,2928.0,Non-First Floor,Resale,Central Region,46,5453566.0,0.31,...,Geylang,14,40,2021 4Q,12,2021,508.91,1.33296,103.894116,657.96
2,2018-09-25,VERTEX,UBI AVENUE 3,1227.0,Non-First Floor,Resale,Central Region,49,5638676.0,1.11,...,Geylang,14,40,2018 3Q,9,2018,485.00,1.33296,103.894116,657.96
3,2014-02-27,VERTEX,UBI AVENUE 3,1421.0,Non-First Floor,Resale,Central Region,53,5469724.0,0.15,...,Geylang,14,40,2014 1Q,2,2014,530.00,1.33296,103.894116,657.96
4,2019-01-29,VERTEX,UBI AVENUE 3,1421.0,Non-First Floor,Resale,Central Region,48,5703569.0,1.90,...,Geylang,14,40,2019 1Q,1,2019,464.48,1.33296,103.894116,657.96
